# 1. Importacion de las librerias

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path  

# 2. Lectura del archivo de datos

In [3]:
data_path = Path("../../data/train.csv")
df = pd.read_csv(data_path)
print(df)

        Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0        1          60       RL         65.0     8450   Pave   NaN      Reg   
1        2          20       RL         80.0     9600   Pave   NaN      Reg   
2        3          60       RL         68.0    11250   Pave   NaN      IR1   
3        4          70       RL         60.0     9550   Pave   NaN      IR1   
4        5          60       RL         84.0    14260   Pave   NaN      IR1   
...    ...         ...      ...          ...      ...    ...   ...      ...   
1455  1456          60       RL         62.0     7917   Pave   NaN      Reg   
1456  1457          20       RL         85.0    13175   Pave   NaN      Reg   
1457  1458          70       RL         66.0     9042   Pave   NaN      Reg   
1458  1459          20       RL         68.0     9717   Pave   NaN      Reg   
1459  1460          20       RL         75.0     9937   Pave   NaN      Reg   

     LandContour Utilities  ... PoolArea PoolQC  Fe

# 3. Validacion de las variables con valores nulos y su cantidad respectiva

Se identificaron 19 variables con valores nulos, siendo las mas criticas PoolQC, MiscFeature, Alley y Fence.

In [4]:
# Extraemos solo las columnas con nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print(f'Total columnas con nulos: {len(nulos)}')
print(nulos)

Total columnas con nulos: 19
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


# 4. Identificación del tipo de nulos alojados en las variables identificadas

Se definieron 2 tipos de nulos, los estructurales, que se refieren a la ausencia de una característica en la propiedad, y los reales por falta de información, que se refieren a la ausencia de datos en una variable que debería tener un valor registrado.

In [5]:
clasificacion_nulos = {
    'PoolQC':       'estructural',  # NA = No Pool
    'LotFrontage':  'real',         
    'MiscFeature':  'estructural',  # NA = None
    'Alley':        'estructural',  # NA = No alley access
    'Fence':        'estructural',  # NA = No Fence
    'MasVnrType':   'estructural',  # None = None
    'FireplaceQu':  'estructural',  # NA = No Fireplace
    'GarageType':   'estructural',  # NA = No Garage
    'GarageYrBlt':  'estructural',  # NA = No Garage       
    'GarageFinish': 'estructural',  # NA = No Garage
    'GarageQual':   'estructural',  # NA = No Garage
    'GarageCond':   'estructural',  # NA = No Garage
    'BsmtExposure': 'estructural',  # NA = No Basement
    'BsmtFinType2': 'estructural',  # NA = No Basement
    'BsmtQual':     'estructural',  # NA = No Basement
    'BsmtCond':     'estructural',  # NA = No Basement
    'BsmtFinType1': 'estructural',  # NA = No Basement
    'MasVnrArea':   'real',         
    'Electrical':   'real',         
}

# 5. Transformacion de los nulos estructurales a un valor representativo

Para poder mapear plenamente los nulos estructurales, se asigno eel valor "None" a cada una de las variables con el fin de representar la ausencia de la característica en la propiedad. En el caso de YearBuilt, se asigno el valor 0 para representar la ausencia de una fecha de construcción registrada.

In [6]:
categoricas_estructurales = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'BsmtExposure', 'BsmtFinType2', 'BsmtQual',
    'BsmtCond', 'BsmtFinType1'
]

for col in categoricas_estructurales:
    df[col] = df[col].fillna('None')
    
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

nulos_post = df[categoricas_estructurales + ['GarageYrBlt']].isnull().sum()
print("Nulos restantes en variables estructurales:")
print(nulos_post[nulos_post > 0])

Nulos restantes en variables estructurales:
Series([], dtype: int64)


# 6. Transformacion de los nulos reales a un valor representativo

1. En el caso de LotFrontage, se asigno el valor de la mediana para cada barrio. Esto determinado por las caracteristicas de las propiedades segun su vecindario y para evitar sobreestimaciones al utilizar la mediana o la media global.
2. En el caso de MasVnrArea, se asigno el valor de la mediana global dada la baja cantidad de nulos.
3. En el caso de Electrical, se asigno el valor de la moda global dada la baja cantidad de nulos.

In [ ]:
# LotFrontage → mediana agrupada por Neighborhood
# transform('median') calcula la mediana por grupo y la asigna
# a cada fila según su vecindario
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Son solo 8 nulos - no justifica agrupación
df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())

# Variable categórica - se imputa con el valor más frecuente
# mode()[0] extrae el primer valor de la moda
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [8]:
# Verificamos que no queden nulos en todo el dataset
total_nulos = df.isnull().sum().sum()
print(f'Total nulos restantes en el dataset: {total_nulos}')

Total nulos restantes en el dataset: 0


# 7. Almacenamiento del dataset limpio para su posterior uso

Almacenamiento del DF en un archivos CSV para su posterior uso en el proceso de modelado.

In [33]:
# Guardar en /data al nivel raíz del proyecto
clean_path = Path('../../data/train_clean.csv')
clean_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(clean_path, index=False)

print(f'Dataset limpio guardado en: {clean_path.resolve()}')
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')

Dataset limpio guardado en: C:\Users\ivanf\Documents\Vortex\1.4 Vortex - House Prices\data\train_clean.csv
Filas: 1460
Columnas: 81
Nulos totales: 0
